In [22]:
using LowLevelFEM
using LinearAlgebra

In [23]:
openGeometry("pipe.geo")

#openPreProcessor()

In [24]:
mat = Material("pipe")

P = Field([mat], type=:ScalarField, fieldName=:p, rhsName=:fp, dim=2, reducedOrder=true)

V = Field([mat], type=:VectorField, fieldName=:v, rhsName=:fv, dim=2);

In [25]:
ndofs(V)

27342

In [26]:
uMax = 1.0

u_in(x, y, z) = uMax * (1.0 - ((y - 0.5) / 0.5)^2);

In [27]:
inlet = BoundaryCondition("left", field=V, vx=u_in, vy=0.0)

bottom = BoundaryCondition("bottom", field=V, vx=0.0, vy=0.0)

top = BoundaryCondition("top", field=V, vx=0.0, vy=0.0)

obstacle = BoundaryCondition("obstacle", field=V, vx=0.0, vy=0.0)

pref = BoundaryCondition("reference", field=P, p=0.0);

In [28]:
μ = 0.1

A = ∫((Grad(V) ⋅ Grad(V)) * μ; Ω="pipe")

B = ∫(Div(V) ⋅ P; Ω="pipe")

C = 0 * ∫(P ⋅ P; Ω="pipe");

In [29]:
uk = VectorField(V, "pipe", [0, 0, 0])
uk = projectTo2D(uk);

In [30]:
tol = 1e-8
maxiter = 20

uk = VectorField(V, "pipe", [0, 0, 0])
uk = projectTo2D(uk)

unew = nothing
pnew = nothing

for k in 1:maxiter

    # ------------------------------------------------------------
    # Newton tangent: (uᵏ · ∇)u
    # ------------------------------------------------------------
    M = [
        uk[1] uk[2] 0 0
        0 0 uk[1] uk[2]
    ]

    N1 = ∫(V ⋅ M ⋅ Grad(V); Ω="pipe")

    # ------------------------------------------------------------
    # Newton tangent: (u · ∇)uᵏ
    # ------------------------------------------------------------
    J = [
        ∂x(uk[1]) ∂y(uk[1])
        ∂x(uk[2]) ∂y(uk[2])
    ]

    N2 = ∫(V ⋅ J ⋅ V; Ω="pipe")

    Av = A + N1 + N2

    # ------------------------------------------------------------
    # RHS: (uᵏ · ∇)uᵏ
    # ------------------------------------------------------------
    rx = uk[1] * ∂x(uk[1]) + uk[2] * ∂y(uk[1])
    ry = uk[1] * ∂x(uk[2]) + uk[2] * ∂y(uk[2])

    loadN = LoadCondition("pipe", field=V, fvx=rx, fvy=ry)

    fv = loadVector(V, [loadN])
    fp = loadVector(P, [])

    F = SystemVector([fv, fp])

    # ------------------------------------------------------------
    # Coupled Newton system
    # ------------------------------------------------------------
    K = SystemMatrix([
        Av -B
        -B' C
    ])

    unew, pnew = solveField(K, F, support=[inlet, bottom, top, obstacle, pref])

    # ------------------------------------------------------------
    # Convergence
    # ------------------------------------------------------------
    uk = elementsToNodes(uk)
    err = norm(unew.a - uk.a) / max(norm(unew.a), eps())

    println("Newton iteration $k:  error = $err")

    uk = unew

    err < tol && break
end

u = unew
p = pnew;

Newton iteration 1:  error = 1.0
Newton iteration 2:  error = 0.03898257378132564
Newton iteration 3:  error = 0.00011907389096186253
Newton iteration 4:  error = 5.7334386815101817e-8
Newton iteration 5:  error = 5.718181304996078e-10


In [31]:
uMax = 5.0

u_in(x, y, z) = uMax * (1.0 - ((y - 0.5) / 0.5)^2);

In [32]:
inlet = BoundaryCondition("left", field=V, vx=u_in, vy=0.0)

bottom = BoundaryCondition("bottom", field=V, vx=0.0, vy=0.0)

top = BoundaryCondition("top", field=V, vx=0.0, vy=0.0)

obstacle = BoundaryCondition("obstacle", field=V, vx=0.0, vy=0.0)

pref = BoundaryCondition("reference", field=P, p=0.0);

In [33]:
tol = 1e-8
maxiter = 20

uk = u

unew = nothing
pnew = nothing

for k in 1:maxiter

    # ------------------------------------------------------------
    # Newton tangent: (uᵏ · ∇)u
    # ------------------------------------------------------------
    M = [
        uk[1] uk[2] 0 0
        0 0 uk[1] uk[2]
    ]

    N1 = ∫(V ⋅ M ⋅ Grad(V); Ω="pipe")

    # ------------------------------------------------------------
    # Newton tangent: (u · ∇)uᵏ
    # ------------------------------------------------------------
    J = [
        ∂x(uk[1]) ∂y(uk[1])
        ∂x(uk[2]) ∂y(uk[2])
    ]

    N2 = ∫(V ⋅ J ⋅ V; Ω="pipe")

    Av = A + N1 + N2

    # ------------------------------------------------------------
    # RHS: (uᵏ · ∇)uᵏ
    # ------------------------------------------------------------
    rx = uk[1] * ∂x(uk[1]) + uk[2] * ∂y(uk[1])
    ry = uk[1] * ∂x(uk[2]) + uk[2] * ∂y(uk[2])

    loadN = LoadCondition("pipe", field=V, fvx=rx, fvy=ry)

    fv = loadVector(V, [loadN])
    fp = loadVector(P, [])

    F = SystemVector([fv, fp])

    # ------------------------------------------------------------
    # Coupled Newton system
    # ------------------------------------------------------------
    K = SystemMatrix([
        Av -B
        -B' C
    ])

    unew, pnew = solveField(K, F, support=[inlet, bottom, top, obstacle, pref])

    # ------------------------------------------------------------
    # Convergence
    # ------------------------------------------------------------
    uk = elementsToNodes(uk)
    err = norm(unew.a - uk.a) / max(norm(unew.a), eps())

    println("Newton iteration $k:  error = $err")

    uk = unew

    err < tol && break
end

u = unew
p = pnew;

Newton iteration 1:  error = 0.8002660542966067
Newton iteration 2:  error = 0.10583387463357673
Newton iteration 3:  error = 0.005436145771128233
Newton iteration 4:  error = 3.339558263443434e-5
Newton iteration 5:  error = 1.5263825413294663e-7
Newton iteration 6:  error = 1.1025913528586716e-8
Newton iteration 7:  error = 6.994277355702736e-10


In [34]:
uMax = 10.0

u_in(x, y, z) = uMax * (1.0 - ((y - 0.5) / 0.5)^2);

In [35]:
inlet = BoundaryCondition("left", field=V, vx=u_in, vy=0.0)

bottom = BoundaryCondition("bottom", field=V, vx=0.0, vy=0.0)

top = BoundaryCondition("top", field=V, vx=0.0, vy=0.0)

obstacle = BoundaryCondition("obstacle", field=V, vx=0.0, vy=0.0)

pref = BoundaryCondition("reference", field=P, p=0.0);

In [36]:
tol = 1e-8
maxiter = 20

uk = u

unew = nothing
pnew = nothing

for k in 1:maxiter

    # ------------------------------------------------------------
    # Newton tangent: (uᵏ · ∇)u
    # ------------------------------------------------------------
    M = [
        uk[1] uk[2] 0 0
        0 0 uk[1] uk[2]
    ]

    N1 = ∫(V ⋅ M ⋅ Grad(V); Ω="pipe")

    # ------------------------------------------------------------
    # Newton tangent: (u · ∇)uᵏ
    # ------------------------------------------------------------
    J = [
        ∂x(uk[1]) ∂y(uk[1])
        ∂x(uk[2]) ∂y(uk[2])
    ]

    N2 = ∫(V ⋅ J ⋅ V; Ω="pipe")

    Av = A + N1 + N2

    # ------------------------------------------------------------
    # RHS: (uᵏ · ∇)uᵏ
    # ------------------------------------------------------------
    rx = uk[1] * ∂x(uk[1]) + uk[2] * ∂y(uk[1])
    ry = uk[1] * ∂x(uk[2]) + uk[2] * ∂y(uk[2])

    loadN = LoadCondition("pipe", field=V, fvx=rx, fvy=ry)

    fv = loadVector(V, [loadN])
    fp = loadVector(P, [])

    F = SystemVector([fv, fp])

    # ------------------------------------------------------------
    # Coupled Newton system
    # ------------------------------------------------------------
    K = SystemMatrix([
        Av -B
        -B' C
    ])

    unew, pnew = solveField(K, F, support=[inlet, bottom, top, obstacle, pref])

    # ------------------------------------------------------------
    # Convergence
    # ------------------------------------------------------------
    uk = elementsToNodes(uk)
    err = norm(unew.a - uk.a) / max(norm(unew.a), eps())

    println("Newton iteration $k:  error = $err")

    uk = unew

    err < tol && break
end

u = unew
p = pnew;

Newton iteration 1:  error = 0.5064389372392646
Newton iteration 2:  error = 0.0755453294609178
Newton iteration 3:  error = 0.005716541456363348
Newton iteration 4:  error = 5.178119222226221e-5
Newton iteration 5:  error = 8.07764848217134e-7
Newton iteration 6:  error = 1.1282392093633963e-7
Newton iteration 7:  error = 8.7106326115709e-9


In [37]:
uMax = 25.0

u_in(x, y, z) = uMax * (1.0 - ((y - 0.5) / 0.5)^2);

In [38]:
inlet = BoundaryCondition("left", field=V, vx=u_in, vy=0.0)

bottom = BoundaryCondition("bottom", field=V, vx=0.0, vy=0.0)

top = BoundaryCondition("top", field=V, vx=0.0, vy=0.0)

obstacle = BoundaryCondition("obstacle", field=V, vx=0.0, vy=0.0)

pref = BoundaryCondition("reference", field=P, p=0.0);

In [39]:
tol = 1e-8
maxiter = 20

uk = u

unew = nothing
pnew = nothing

for k in 1:maxiter

    # ------------------------------------------------------------
    # Newton tangent: (uᵏ · ∇)u
    # ------------------------------------------------------------
    M = [
        uk[1] uk[2] 0 0
        0 0 uk[1] uk[2]
    ]

    N1 = ∫(V ⋅ M ⋅ Grad(V); Ω="pipe")

    # ------------------------------------------------------------
    # Newton tangent: (u · ∇)uᵏ
    # ------------------------------------------------------------
    J = [
        ∂x(uk[1]) ∂y(uk[1])
        ∂x(uk[2]) ∂y(uk[2])
    ]

    N2 = ∫(V ⋅ J ⋅ V; Ω="pipe")

    Av = A + N1 + N2

    # ------------------------------------------------------------
    # RHS: (uᵏ · ∇)uᵏ
    # ------------------------------------------------------------
    rx = uk[1] * ∂x(uk[1]) + uk[2] * ∂y(uk[1])
    ry = uk[1] * ∂x(uk[2]) + uk[2] * ∂y(uk[2])

    loadN = LoadCondition("pipe", field=V, fvx=rx, fvy=ry)

    fv = loadVector(V, [loadN])
    fp = loadVector(P, [])

    F = SystemVector([fv, fp])

    # ------------------------------------------------------------
    # Coupled Newton system
    # ------------------------------------------------------------
    K = SystemMatrix([
        Av -B
        -B' C
    ])

    unew, pnew = solveField(K, F, support=[inlet, bottom, top, obstacle, pref])

    # ------------------------------------------------------------
    # Convergence
    # ------------------------------------------------------------
    uk = elementsToNodes(uk)
    err = norm(unew.a - uk.a) / max(norm(unew.a), eps())

    println("Newton iteration $k:  error = $err")

    uk = unew

    err < tol && break
end

u = unew
p = pnew;

Newton iteration 1:  error = 0.6185127270665317
Newton iteration 2:  error = 0.2786750210204008
Newton iteration 3:  error = 0.10874101933703946
Newton iteration 4:  error = 0.10720335692525562
Newton iteration 5:  error = 0.02348375600554924
Newton iteration 6:  error = 0.004168940058271105
Newton iteration 7:  error = 4.0683577151924746e-5
Newton iteration 8:  error = 6.263510411820273e-6
Newton iteration 9:  error = 1.6120410803467602e-6
Newton iteration 10:  error = 3.5069419935729015e-7
Newton iteration 11:  error = 1.2006130638399808e-7
Newton iteration 12:  error = 4.711994627744568e-8
Newton iteration 13:  error = 1.8173504669437758e-8
Newton iteration 14:  error = 1.0313397462427283e-8
Newton iteration 15:  error = 3.825976998100723e-9


In [40]:
showDoFResults(u, name="v")
showDoFResults(p, name="p");

In [41]:
divu = ∂x(u[1]) + ∂y(u[2])
showDoFResults(divu, name="div(v)");

In [42]:
openPostProcessor();